# 04 - Build SE-HC

This notebook documents and inspects the construction of the final thesis-facing SE-HC ensemble, implemented as `SIGMA_FINAL`.


## Purpose of this notebook

This notebook builds the final SE-HC ensemble. The inputs are OOF prediction candidates generated in the previous notebook, and the output is the final thesis-facing SE-HC / `SIGMA_FINAL` prediction.

SE-HC uses Caruana-style hill-climbing to select and average candidate prediction vectors. This is an ensemble over predictions, not a new model trained directly on raw features.


## From baseline/candidate OOFs to ensemble library

Baseline models are used for algorithm-level comparison. Candidate predictors are OOF prediction vectors used for ensemble construction. Candidate predictors may come from V15 models, historical strong models, or stacked predictors.

The candidate library below matches the original `SIGMA_FINAL` construction in `notebooks/archive/solution_ver02.ipynb`, cell 209. Full OOF parquet files are local-only artifacts.


In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
OOF_DIR = ROOT / 'outputs' / 'oof_predictions'
ID_COL = 'SK_ID_CURR'
TARGET = 'TARGET'

CANDIDATE_SPECS = {
    'fp_final': ('oof_FP_FINAL.parquet', 'oof_final', 'SPC / Stacked Prediction Candidate'),
    'lgb_v15_ms': ('oof_lgb_v15_multiseed.parquet', 'oof_lgb', 'V15 Multi-Seed LightGBM; final SE-HC component'),
    'lgb_v12_ms': ('oof_lgb_v12_multiseed.parquet', 'oof_lgb', 'historical LightGBM candidate'),
    'lgb_v11_ms': ('oof_lgb_v11_multiseed.parquet', 'oof_lgb', 'historical LightGBM candidate'),
    'lgb_v7_ms': ('oof_lgb_v7_multiseed.parquet', 'oof_lgb', 'historical LightGBM candidate'),
    'cb_v11_ms': ('oof_cb_v11_multiseed.parquet', 'oof_cb', 'historical CatBoost multi-seed candidate'),
    'xgb_v11': ('oof_xgb_v11.parquet', 'oof_xgb', 'historical XGBoost candidate'),
    'mlp2': ('oof_mlp2.parquet', 'oof_mlp', 'historical neural-network candidate'),
    'lgb_C': ('oof_lgb_C.parquet', 'oof_lgb', 'historical regularized LightGBM candidate'),
    'stack_v8': ('oof_K3_stack_v8_FINAL.parquet', 'oof_final', 'historical stacked predictor'),
}

def prediction_column(df: pd.DataFrame) -> str:
    return [c for c in df.columns if c not in [ID_COL, TARGET]][0]

def load_oof_library():
    rows = []
    candidates_oof = {}
    reference = None
    for name, (filename, expected_col, role) in CANDIDATE_SPECS.items():
        path = OOF_DIR / filename
        if not path.exists():
            rows.append({'candidate': name, 'role': role, 'status': 'MISSING', 'oof_path': str(path.relative_to(ROOT))})
            continue
        df = pd.read_parquet(path)
        pred_col = expected_col if expected_col in df.columns else prediction_column(df)
        if reference is None:
            reference = df[[ID_COL, TARGET]].copy()
        else:
            if not df[ID_COL].equals(reference[ID_COL]) or not df[TARGET].equals(reference[TARGET]):
                raise ValueError(f'{name} is not aligned on SK_ID_CURR/TARGET')
        pred = df[pred_col].astype(float).to_numpy()
        candidates_oof[name] = pred
        rows.append({
            'candidate': name,
            'role': role,
            'oof_path': str(path.relative_to(ROOT)),
            'prediction_column': pred_col,
            'shape': f'{df.shape[0]} x {df.shape[1]}',
            'oof_roc_auc': roc_auc_score(df[TARGET].astype(int), pred),
            'status': 'OK',
        })
    y = reference[TARGET].astype(int).to_numpy() if reference is not None else None
    return pd.DataFrame(rows), candidates_oof, y

candidate_library, candidates_oof, y = load_oof_library()
candidate_library


 candidate                                           role                                              oof_path prediction_column      shape  oof_roc_auc status
  fp_final             SPC / Stacked Prediction Candidate          outputs/oof_predictions/oof_FP_FINAL.parquet         oof_final 307507 x 3     0.799879     OK
lgb_v15_ms V15 Multi-Seed LightGBM; final SE-HC component outputs/oof_predictions/oof_lgb_v15_multiseed.parquet           oof_lgb 307507 x 3     0.799749     OK
lgb_v12_ms                  historical LightGBM candidate outputs/oof_predictions/oof_lgb_v12_multiseed.parquet           oof_lgb 307507 x 3     0.798544     OK
lgb_v11_ms                  historical LightGBM candidate outputs/oof_predictions/oof_lgb_v11_multiseed.parquet           oof_lgb 307507 x 3     0.797993     OK
 lgb_v7_ms                  historical LightGBM candidate  outputs/oof_predictions/oof_lgb_v7_multiseed.parquet           oof_lgb 307507 x 3     0.797122     OK
 cb_v11_ms       historical CatBoo

## SPC building

SPC stands for **Stacked Prediction Candidate**. It corresponds to `fp_final` in the code.

SPC is an intermediate stacked/meta prediction candidate created before the final SE-HC step. It is not the same as the final SE-HC model. In SE - HC, SPC is used as one candidate in the hill-climbing library.


In [ ]:
candidate_library[candidate_library['candidate'].eq('fp_final')]


candidate                               role                                     oof_path prediction_column      shape  oof_roc_auc status
 fp_final SPC / Stacked Prediction Candidate outputs/oof_predictions/oof_FP_FINAL.parquet         oof_final 307507 x 3     0.799879     OK

## Candidate AUC inspection

Candidate AUC is used only as an initial diagnostic. The best individual candidate is not necessarily the best final model. The final ensemble is selected based on whether combining candidates improves OOF ROC-AUC.


In [ ]:

candidate_library.sort_values('oof_roc_auc', ascending=False, na_position='last')[
    ['candidate', 'role', 'oof_roc_auc', 'shape', 'status']
]


 candidate                                           role  oof_roc_auc      shape status
  fp_final             SPC / Stacked Prediction Candidate     0.799879 307507 x 3     OK
  stack_v8                   historical stacked predictor     0.799751 307507 x 3     OK
lgb_v15_ms V15 Multi-Seed LightGBM; final SE-HC component     0.799749 307507 x 3     OK
lgb_v12_ms                  historical LightGBM candidate     0.798544 307507 x 3     OK
lgb_v11_ms                  historical LightGBM candidate     0.797993 307507 x 3     OK
 lgb_v7_ms                  historical LightGBM candidate     0.797122 307507 x 3     OK
 cb_v11_ms       historical CatBoost multi-seed candidate     0.796316 307507 x 3     OK
   xgb_v11                   historical XGBoost candidate     0.795867 307507 x 3     OK
      mlp2            historical neural-network candidate     0.780671 307507 x 3     OK
     lgb_C      historical regularized LightGBM candidate     0.760278 307507 x 3     OK

## Caruana hill-climbing method

Algorithm:

1. Start with an empty ensemble.
2. At each iteration, test adding each candidate to the current average.
3. Select the candidate that gives the highest OOF ROC-AUC improvement.
4. Stop when no candidate improves the ensemble.
5. Candidate frequency corresponds to ensemble weight.

The core function below is copied from the original `SIGMA_FINAL` construction cell with no change to the greedy selection logic.


In [ ]:
def caruana_hill_climb(candidates_oof, y, max_iters=50, verbose=True):
    """Greedy forward selection with replacement."""
    ensemble = []
    ensemble_pred = np.zeros(len(y))

    for iteration in range(max_iters):
        current_auc = roc_auc_score(y, ensemble_pred) if iteration > 0 else 0

        best_auc = current_auc
        best_model = None

        for name, pred in candidates_oof.items():
            if iteration == 0:
                new_pred = pred.copy()
            else:
                new_pred = (ensemble_pred * iteration + pred) / (iteration + 1)

            auc = roc_auc_score(y, new_pred)

            if auc > best_auc:
                best_auc = auc
                best_model = name

        if best_model is None:
            if verbose:
                print(f'  Iter {iteration}: No improvement, stopping')
            break

        ensemble.append(best_model)
        if iteration == 0:
            ensemble_pred = candidates_oof[best_model].copy()
        else:
            ensemble_pred = (ensemble_pred * iteration + candidates_oof[best_model]) / (iteration + 1)

        if verbose:
            print(f'  Iter {iteration+1:2d}: +{best_model:15s} | AUC = {best_auc:.5f}')

    return ensemble, ensemble_pred


### Optional hill-climb rerun

This cell reruns only the OOF-level hill climb from saved local candidate predictions. It does not retrain base models. Leave it unexecuted during the demo unless local OOF artifacts are available.


In [ ]:

ensemble_models, final_oof = caruana_hill_climb(candidates_oof, y, max_iters=50, verbose=True)
final_auc = roc_auc_score(y, final_oof)
model_frequencies = pd.Series(ensemble_models).value_counts().rename_axis('candidate').reset_index(name='frequency')
model_frequencies['weight'] = model_frequencies['frequency'] / len(ensemble_models)
print(f'Final AUC: {final_auc:.6f}')
model_frequencies


  Iter  1: +fp_final        | AUC = 0.79988
  Iter  2: +lgb_v15_ms      | AUC = 0.80169
  Iter 2: No improvement, stopping
Final AUC: 0.801688


 candidate  frequency  weight
  fp_final          1     0.5
lgb_v15_ms          1     0.5

## Final SE-HC result

Final SE-HC selected:

- SPC / `fp_final` with weight `0.5`
- V15 Multi-Seed LightGBM / `lgb_v15_ms` with weight `0.5`

Formula:

`SE-HC = 0.5 * SPC + 0.5 * V15 Multi-Seed LightGBM`

OOF ROC-AUC: **0.801688**

The table below records the final selected candidates, their frequencies/weights from the Caruana selection, the final OOF score, and the saved local artifact paths. SPC is a selected candidate, not the final model by itself.


In [ ]:

final_selected_components = pd.DataFrame([
    {
        'candidate': 'fp_final',
        'thesis_name': 'SPC / Stacked Prediction Candidate',
        'frequency': 1,
        'weight': 0.5,
        'role': 'intermediate stacked prediction candidate',
    },
    {
        'candidate': 'lgb_v15_ms',
        'thesis_name': 'V15 Multi-Seed LightGBM',
        'frequency': 1,
        'weight': 0.5,
        'role': 'strong V15 tree model candidate',
    },
])

final_se_hc_summary = {
    'implementation_name': 'SIGMA_FINAL',
    'thesis_name': 'SE-HC / Stacked Ensemble with Hill-Climbing Selection',
    'formula': 'SE-HC = 0.5 * SPC + 0.5 * V15 Multi-Seed LightGBM',
    'oof_roc_auc': 0.8016882969488922,
    'saved_oof_path': 'outputs/oof_predictions/oof_SIGMA_FINAL.parquet',
    'saved_metrics_path': 'evaluation/metrics/metrics_SIGMA_FINAL.json',
    'saved_deciles_path': 'evaluation/deciles/deciles_SIGMA_FINAL.csv',
    'saved_dashboard_path': 'evaluation/figures outputs/dashboard_SIGMA_FINAL.png',
}

display(final_selected_components)
final_se_hc_summary


   candidate                         thesis_name  frequency  weight                                  role
0   fp_final  SPC / Stacked Prediction Candidate          1     0.5  intermediate stacked prediction candidate
1  lgb_v15_ms             V15 Multi-Seed LightGBM          1     0.5               strong V15 tree model candidate

{'implementation_name': 'SIGMA_FINAL',
 'thesis_name': 'SE-HC / Stacked Ensemble with Hill-Climbing Selection',
 'formula': 'SE-HC = 0.5 * SPC + 0.5 * V15 Multi-Seed LightGBM',
 'oof_roc_auc': 0.8016882969488922,
 'saved_oof_path': 'outputs/oof_predictions/oof_SIGMA_FINAL.parquet',
 'saved_metrics_path': 'evaluation/metrics/metrics_SIGMA_FINAL.json',
 'saved_deciles_path': 'evaluation/deciles/deciles_SIGMA_FINAL.csv',
 'saved_dashboard_path': 'evaluation/figures outputs/dashboard_SIGMA_FINAL.png'}

## Stability check

This checks whether the final saved OOF prediction remains stable under different fold partitions. It is not retraining the model; it evaluates the saved `SIGMA_FINAL` OOF prediction under different split views.


In [ ]:

from sklearn.model_selection import StratifiedKFold

sigma_oof = pd.read_parquet(OOF_DIR / 'oof_SIGMA_FINAL.parquet')
sigma_col = prediction_column(sigma_oof)
y_sigma = sigma_oof[TARGET].astype(int).to_numpy()
pred_sigma = sigma_oof[sigma_col].astype(float).to_numpy()

rows = []
for cv_seed in [42, 7, 99]:
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=cv_seed)
    fold_aucs = []
    for _, val_idx in skf.split(np.zeros(len(y_sigma)), y_sigma):
        fold_aucs.append(roc_auc_score(y_sigma[val_idx], pred_sigma[val_idx]))
    rows.append({
        'cv_seed': cv_seed,
        'mean_auc': np.mean(fold_aucs),
        'std_auc': np.std(fold_aucs),
        'fold_aucs': [round(v, 6) for v in fold_aucs],
    })

pd.DataFrame(rows)


 cv_seed  mean_auc  std_auc                                          fold_aucs
      42  0.801698 0.004036 [0.797971, 0.808881, 0.798574, 0.803287, 0.799777]
       7  0.801737 0.002005 [0.798914, 0.801565, 0.804158, 0.800282, 0.803767]
      99  0.801736 0.001940  [0.800859, 0.800383, 0.800253, 0.801705, 0.80548]

##  Saved artifacts and handoff to evaluation

Full OOF/submission artifacts are local-only and are not committed to GitHub. Lightweight copies are stored under `reports/results/` and `reports/figures/` for demo.


In [ ]:

artifact_paths = [
    ('Final OOF predictions', 'outputs/oof_predictions/oof_SIGMA_FINAL.parquet', 'local-only'),
    ('Final submission', 'outputs/oof_predictions/submission_SIGMA_FINAL.csv', 'local-only; not found in current canonical folder'),
    ('Final metrics', 'evaluation/metrics/metrics_SIGMA_FINAL.json', 'local-only source'),
    ('Final deciles', 'evaluation/deciles/deciles_SIGMA_FINAL.csv', 'local-only source'),
    ('Final dashboard', 'evaluation/figures outputs/dashboard_SIGMA_FINAL.png', 'local-only source'),
    ('SIGMA summary', 'outputs/others/PLAN_SIGMA_SUMMARY.json', 'local-only source'),
    ('Demo metrics copy', 'reports/results/metrics_SE_HC.json', 'GitHub-lightweight'),
    ('Demo deciles copy', 'reports/results/deciles_SE_HC.csv', 'GitHub-lightweight'),
    ('Demo dashboard copy', 'reports/figures/dashboard_SE_HC.png', 'GitHub-lightweight'),
]
rows = []
for artifact, rel_path, policy in artifact_paths:
    path = ROOT / rel_path
    rows.append({
        'artifact': artifact,
        'path': rel_path,
        'exists': path.exists(),
        'size_bytes': path.stat().st_size if path.exists() else None,
        'policy': policy,
    })
pd.DataFrame(rows)


             artifact                                                 path  exists  size_bytes                                            policy
Final OOF predictions      outputs/oof_predictions/oof_SIGMA_FINAL.parquet    True   4596948.0                                        local-only
     Final submission   outputs/oof_predictions/submission_SIGMA_FINAL.csv   False         NaN local-only; not found in current canonical folder
        Final metrics          evaluation/metrics/metrics_SIGMA_FINAL.json    True       378.0                                 local-only source
        Final deciles           evaluation/deciles/deciles_SIGMA_FINAL.csv    True       382.0                                 local-only source
      Final dashboard evaluation/figures outputs/dashboard_SIGMA_FINAL.png    True    103359.0                                 local-only source
        SIGMA summary               outputs/others/PLAN_SIGMA_SUMMARY.json    True       426.0                                 loc